# Smart Traffic — Congestion Classifier (Google Colab)

Trains the **congestion** model on the
[Norway traffic-camera dataset](https://huggingface.co/datasets/ilsilfverskiold/traffic-camera-norway-images)
(6,782 labelled images) with **EfficientNetV2-S transfer learning**, evaluates it, and
saves a `traffic.pt` you drop into `services/ml/models/` — the ML service loads it
automatically (the checkpoint stores its own `arch` + `classes`).

**Before running:** menu → *Runtime → Change runtime type → GPU*.

This notebook is self-contained: it pulls the dataset straight from Hugging Face onto
Colab (nothing touches your PC, nothing goes into GitHub) and only the small `.pt` comes
back. Just run every cell top to bottom.

Label mapping (by **name** — the dataset's integer ids are not ordinal):

| dataset class | our class | weight |
|---|---|---|
| no-traffic | empty | 1 |
| low-traffic | low | 2 |
| medium-traffic | high | 3 |
| high-traffic | jam | 4 |


### 1. Install
Colab already ships torch + torchvision (with CUDA); we only add `datasets`.

In [ ]:
!pip -q install datasets

### 2. Imports + configuration
Change `EPOCHS` or `ARCH` here if you like (`efficientnet` = best, `mobilenet` = tiny ~6 MB).

In [ ]:
import json, time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU for a fast run.")

REPO = "ilsilfverskiold/traffic-camera-norway-images"

# our ordinal taxonomy + weights (must match the service: app/config.py, types.ts)
CLASSES = ["empty", "low", "high", "jam"]
WEIGHTS = {"empty": 1, "low": 2, "high": 3, "jam": 4}
W_MAX = 4
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# dataset class NAME -> our class name (ids are non-ordinal, so map by name)
LABEL_MAP = {
    "no-traffic": "empty",
    "low-traffic": "low",
    "medium-traffic": "high",
    "high-traffic": "jam",
}

IMG_SIZE = 224
BATCH = 32
EPOCHS = 12
ARCH = "efficientnet"   # "efficientnet" | "mobilenet"
OUT = "traffic.pt"

### 3. Load the dataset + build data loaders
We wrap the Hugging Face splits directly (no need to write 6k files), remap labels, and apply ImageNet-style transforms with light augmentation on the training split.

In [ ]:
raw = load_dataset(REPO)
print(raw)
int2str = raw["train"].features["label"].int2str

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class HFTraffic(Dataset):
    def __init__(self, split, tf):
        self.ds = raw[split]
        self.tf = tf
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        ex = self.ds[i]
        our_name = LABEL_MAP[int2str(ex["label"])]
        y = CLASS_TO_IDX[our_name]
        x = self.tf(ex["image"].convert("RGB"))
        return x, y

train_loader = DataLoader(HFTraffic("train", train_tf), batch_size=BATCH, shuffle=True, num_workers=2)
val_loader = DataLoader(HFTraffic("validation", eval_tf), batch_size=BATCH, num_workers=2)

# class balance -> weighted loss (the dataset is uneven)
counts = [0] * len(CLASSES)
for lab in raw["train"]["label"]:
    counts[CLASS_TO_IDX[LABEL_MAP[int2str(lab)]]] += 1
print("train class counts:", dict(zip(CLASSES, counts)))
total = sum(counts)
class_weights = torch.tensor(
    [total / (len(CLASSES) * c) if c else 0.0 for c in counts],
    dtype=torch.float, device=device,
)

### 4. Build the model (EfficientNetV2-S, pretrained head replaced)

In [ ]:
def build_model(arch, num_classes):
    if arch == "efficientnet":
        net = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
        net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, num_classes)
    elif arch == "mobilenet":
        net = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, num_classes)
    else:
        raise ValueError(arch)
    return net

model = build_model(ARCH, len(CLASSES)).to(device)
print(f"built {ARCH} with {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

### 5. Evaluation helper — accuracy, confusion matrix, per-class P/R/F1, and the weighted-error equation
This mirrors `app/scoring.py` so the numbers match the report.

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    conf = {t: {p: 0 for p in CLASSES} for t in CLASSES}
    n = correct = 0
    sum_err = 0.0
    for x, y in loader:
        x = x.to(device)
        pred = model(x).argmax(1).cpu().tolist()
        for t_idx, p_idx in zip(y.tolist(), pred):
            t, p = CLASSES[t_idx], CLASSES[p_idx]
            conf[t][p] += 1
            n += 1
            correct += 1 if t == p else 0
            sum_err += abs(WEIGHTS[p] - WEIGHTS[t]) / W_MAX
    per_class = {}
    for c in CLASSES:
        tp = conf[c][c]
        fn = sum(conf[c].values()) - tp
        fp = sum(conf[t][c] for t in CLASSES) - tp
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        per_class[c] = {"precision": round(prec, 4), "recall": round(rec, 4),
                        "f1": round(f1, 4), "support": tp + fn}
    return {
        "accuracy": correct / max(n, 1),
        "mean_weighted_error": sum_err / max(n, 1),
        "confusion": conf,
        "per_class": per_class,
        "n": n,
    }

### 6. Train
Fine-tunes for `EPOCHS` epochs with a class-weighted loss and (on GPU) mixed precision.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
use_amp = device == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    t0 = time.time()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type=device, enabled=use_amp):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item()
    val = evaluate(val_loader)
    print(f"epoch {epoch:2d}/{EPOCHS}  loss={running/len(train_loader):.4f}  "
          f"val_acc={val['accuracy']:.4f}  mwe={val['mean_weighted_error']:.4f}  "
          f"({time.time()-t0:.0f}s)")

### 7. Final report on the validation split

In [ ]:
metrics = evaluate(val_loader)
print(f"accuracy            : {metrics['accuracy']:.4f}")
print(f"mean weighted error : {metrics['mean_weighted_error']:.4f}")
print("\nper-class:")
for c in CLASSES:
    m = metrics["per_class"][c]
    print(f"  {c:6s} P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} (n={m['support']})")
print("\nconfusion matrix (rows = true, cols = pred):")
header = "        " + "".join(f"{c:>8s}" for c in CLASSES)
print(header)
for t in CLASSES:
    print(f"{t:>7s} " + "".join(f"{metrics['confusion'][t][p]:>8d}" for p in CLASSES))

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nsaved metrics.json")

### 8. Save the checkpoint (exact format the service loads)

In [ ]:
torch.save(
    {"state_dict": model.state_dict(), "arch": ARCH, "classes": CLASSES},
    OUT,
)
import os
print(f"saved {OUT}  ({os.path.getsize(OUT)/1e6:.1f} MB)  arch={ARCH} classes={CLASSES}")

### 9. (Optional) Export to ONNX
For running inference without Python (e.g. onnxruntime-node inside Bun).

In [ ]:
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
torch.onnx.export(
    model, dummy, "traffic.onnx",
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
print("exported traffic.onnx")

### 10. Download the weights

In [ ]:
from google.colab import files
files.download("traffic.pt")

### Done — what to do with `traffic.pt`

1. Put it in `services/ml/models/traffic.pt` on your machine (that folder is gitignored).
2. Restart the ML service (`uvicorn app.server:app --port 8000`). `/health` now shows
   `traffic: { loaded: true }`.
3. Dashboard → **Classifier** → drop a real intersection image → real verdict.

**Share it (without committing to the repo):**
```bash
gh release create congestion-v1 services/ml/models/traffic.pt -t "Congestion model v1"
```
